# 🔬 LAB 3 — Versão Windows/Local: Ollama + Llama 3.1 8B

**Curso:** MBA RAG & CAG Aplicados a Direito e Segurança Pública  
**Ambiente:** Windows Local (sem Colab)

## Diferenças em relação à versão Colab
| Colab Original | Esta versão |
|---|---|
| vLLM (Linux only) | **Ollama** (Windows nativo) |
| GPU L4 23GB | Quadro P4000 8GB |
| Llama 3.1 8B AWQ | Llama 3.1 8B GGUF Q4 |
| API: localhost:8000 | API: localhost:11434 |

## ⚠️ Pré-requisitos ANTES de executar
1. Instalar Ollama: https://ollama.com/download/windows
2. Baixar o modelo (no terminal): `ollama pull llama3.1:8b`
3. Garantir que o Ollama está rodando (ícone na bandeja do sistema)
4. **Execute as células em ordem — CÉLULA 0 primeiro!**

## 📦 CÉLULA 0 — Instalar Dependências Python

**Execute esta célula primeiro!** Instala `openai`, `requests` e `ipywidgets` no ambiente Python atual do Jupyter.

In [ ]:
# ─── CÉLULA 0: Instalar dependências ────────────────────────────
import subprocess, sys

PACOTES = ['openai', 'requests', 'ipywidgets']

print('📦 Instalando dependências...')
print('=' * 60)

for pacote in PACOTES:
    resultado = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pacote, '--quiet'],
        capture_output=True, text=True
    )
    status = '✅' if resultado.returncode == 0 else '❌'
    print(f'  {status} {pacote}')
    if resultado.returncode != 0:
        print(f'     Erro: {resultado.stderr[:150]}')

print()
print('✅ Pronto! Execute a CÉLULA 1 agora.')

## 📦 CÉLULA 1 — Verificar GPU e Ollama

Verifica a GPU local e testa a conexão com o servidor Ollama.

In [ ]:
# ─── CÉLULA 1: Verificar GPU e conexão com Ollama ───────────────
import subprocess, sys, requests

print('🖥️  VERIFICAÇÃO DE GPU')
print('=' * 60)

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    for linha in result.stdout.strip().split('\n'):
        print(f'  GPU: {linha}')
else:
    print('  ⚠️  nvidia-smi não encontrado (modo CPU)')

print()
print('🦙 VERIFICANDO SERVIDOR OLLAMA')
print('=' * 60)

OLLAMA_URL = 'http://localhost:11434'

try:
    r = requests.get(f'{OLLAMA_URL}/api/tags', timeout=5)
    modelos = [m['name'] for m in r.json().get('models', [])]
    print(f'  ✅ Ollama online!')
    print(f'  Modelos disponíveis: {modelos}')
    if not modelos:
        print('  ⚠️  Nenhum modelo baixado. Execute no terminal: ollama pull llama3.1:8b')
except Exception as e:
    print(f'  ❌ Ollama não está rodando: {e}')
    print('  → Inicie o Ollama pelo ícone na bandeja ou execute: ollama serve')
    raise SystemExit('Inicie o Ollama antes de continuar.')

## 📦 CÉLULA 2 — Configurar Cliente OpenAI → Ollama

O Ollama expõe uma API OpenAI-compatível em `/v1`. O mesmo cliente Python funciona sem alteração!

In [ ]:
# ─── CÉLULA 2: Configurar cliente ───────────────────────────────
from openai import OpenAI

# Ollama usa a mesma interface OpenAI — só muda a URL!
client_vllm = OpenAI(
    base_url=f'{OLLAMA_URL}/v1',
    api_key='ollama'  # qualquer string funciona
)

# Detecta o modelo disponível automaticamente
modelos_disponiveis = client_vllm.models.list()

# Prefere llama3.1, senão pega o primeiro disponível
MODEL_ID = next(
    (m.id for m in modelos_disponiveis.data if 'llama3.1' in m.id or 'llama-3.1' in m.id),
    modelos_disponiveis.data[0].id if modelos_disponiveis.data else None
)

if not MODEL_ID:
    raise SystemExit('Nenhum modelo encontrado. Execute: ollama pull llama3.1:8b')

print(f'✅ Usando modelo: {MODEL_ID}')
print(f'   URL base: {OLLAMA_URL}/v1')
print(f'   Pronto para receber completions!')

## 📦 CÉLULA 3 — Primeiro Completion: Análise Jurídica

Idêntico ao Colab — mesmo código, mesma API `/v1/chat/completions`.

In [ ]:
# ─── CÉLULA 3: Primeiro completion ──────────────────────────────
import time

SYSTEM_PROMPT = (
    'Você é um assistente jurídico especializado em direito penal brasileiro. '
    'Responda sempre de forma técnica e concisa, citando artigos de lei quando aplicável. '
    'NUNCA invente precedentes, súmulas ou artigos inexistentes.'
)

USER_PROMPT = (
    'Qual é a diferença entre os crimes de furto (art. 155 CP) '
    'e roubo (art. 157 CP)? Explique os elementos que os distinguem '
    'e cite um exemplo prático de cada.'
)

print(f'🤖 Usando modelo: {MODEL_ID}')
print('=' * 60)
print(f'📝 Pergunta: {USER_PROMPT[:80]}...')
print('⏳ Aguardando resposta...')
print()

inicio = time.time()

resposta = client_vllm.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': USER_PROMPT}
    ],
    temperature=0.2,
    max_tokens=500,
)

tempo_total     = time.time() - inicio
texto_resposta  = resposta.choices[0].message.content
tokens_prompt   = resposta.usage.prompt_tokens
tokens_resposta = resposta.usage.completion_tokens
tok_por_segundo = tokens_resposta / tempo_total if tempo_total > 0 else 0

print('📊 MÉTRICAS DE GERAÇÃO:')
print(f'   Tokens (prompt):    {tokens_prompt}')
print(f'   Tokens (resposta):  {tokens_resposta}')
print(f'   Tempo total:        {tempo_total:.1f}s')
print(f'   Velocidade:         {tok_por_segundo:.1f} tokens/s')
print()
print('🗒️  RESPOSTA DO MODELO:')
print('-' * 60)
print(texto_resposta)
print('-' * 60)

## 📦 CÉLULA 4 — Experimento: Impacto da Temperature

Mesmo experimento do Colab — compara determinístico vs criativo para análise jurídica.

In [ ]:
# ─── CÉLULA 4: Experimento de temperature ───────────────────────
import sys

PROMPT_CLASSIFICACAO = (
    'Classifique o seguinte crime segundo o Código Penal Brasileiro. '
    'Responda: tipo do crime, Artigo e Pena máxima.\n\n'
    'FATO: "O suspeito subtraiu R$ 500,00 da bolsa da vítima enquanto ela dormia no ônibus."'
)

TEMPERATURAS = [
    (0.0, 'Determinístico — máxima precisão'),
    (0.3, 'Baixo — recomendado para análise jurídica'),
    (0.7, 'Médio — equilíbrio criatividade/precisão'),
    (1.0, 'Alto — criativo, menos adequado para direito'),
]

print('🌡️  EXPERIMENTO: IMPACTO DA TEMPERATURE')
print('=' * 60)

for temp, descricao in TEMPERATURAS:
    print(f'-> Temperature = {temp} ({descricao})')
    inicio = time.time()

    resp = client_vllm.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {'role': 'system', 'content': 'Você é um assistente jurídico técnico e objetivo.'},
            {'role': 'user',   'content': PROMPT_CLASSIFICACAO}
        ],
        temperature=temp,
        max_tokens=80,
    )

    duracao = time.time() - inicio
    texto   = resp.choices[0].message.content.strip()
    print(f'   Resposta: {texto[:200]}')
    print(f'   Tempo: {duracao:.1f}s | Tokens: {resp.usage.completion_tokens}')
    print('-' * 40)
    sys.stdout.flush()

print('\n🚀 Experimento finalizado!')

## 📦 CÉLULA 5 — System Messages: Roleplay Jurídico

O mesmo fato analisado por três personas diferentes — apenas mudando a `system message`.

In [ ]:
# ─── CÉLULA 5: System messages e roleplay jurídico ──────────────
FATO_JURIDICO = (
    'Fato: Acusado foi preso em flagrante com 50g de cocaína dividida em '
    'embalagens individuais. Sem antecedentes criminais. Residência fixa comprovada.'
)

PERSONAS = [
    {
        'nome': '⚖️  JUIZ DE DIREITO',
        'system': (
            'Você é um juiz de direito. Analise os fatos sob a ótica da '
            'necessidade de decretação de prisão preventiva versus medidas cautelares. '
            'Cite apenas art. 319 CPP e 312 CPP.'
        ),
        'pergunta': 'Devo decretar prisão preventiva ou aplicar medidas cautelares alternativas?'
    },
    {
        'nome': '🔍 DELEGADO DE POLÍCIA',
        'system': (
            'Você é um delegado de polícia. Indique as diligências investigativas '
            'prioritárias. Foque em cadeia de custódia e coleta de provas.'
        ),
        'pergunta': 'Quais diligências devo determinar imediatamente para fortalecer o inquérito?'
    },
    {
        'nome': '🧑‍💼 ADVOGADO DE DEFESA',
        'system': (
            'Você é um advogado criminal de defesa. Identifique '
            'as teses defensivas disponíveis. Cite dispositivos legais aplicáveis.'
        ),
        'pergunta': 'Quais são as melhores teses de defesa disponíveis para este caso?'
    },
]

print('🎭 DEMONSTRAÇÃO: SYSTEM MESSAGES PARA PERFIS JURÍDICOS')
print('=' * 60)
print(f'📋 Fato: {FATO_JURIDICO}\n')

for persona in PERSONAS:
    print(f"{' ─ ' * 15}")
    print(f"👤 PERFIL: {persona['nome']}")
    print(f"❓ Pergunta: {persona['pergunta']}")
    print("⏳ Gerando resposta...")

    resp = client_vllm.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {'role': 'system', 'content': persona['system']},
            {'role': 'user',   'content': f"{FATO_JURIDICO}\n\n{persona['pergunta']}"}
        ],
        temperature=0.2,
        max_tokens=300,
    )
    print(f"\n{resp.choices[0].message.content.strip()}\n")

## 📦 CÉLULA 6 — Checklist de Validação

Confirma que todos os objetivos do Lab foram atingidos.

In [ ]:
# ─── CÉLULA 6: Checklist de validação ───────────────────────────
import requests, time

print('✅ CHECKLIST DE VALIDAÇÃO — LAB 3 (Ollama + Llama 3.1 8B)')
print('=' * 65)

checks = []

# Check 1: Ollama respondendo
try:
    r = requests.get(f'{OLLAMA_URL}/api/tags', timeout=5)
    checks.append(('Servidor Ollama ativo', r.status_code == 200, f'HTTP {r.status_code}'))
except Exception as e:
    checks.append(('Servidor Ollama ativo', False, str(e)[:50]))

# Check 2: Modelo disponível
try:
    r = requests.get(f'{OLLAMA_URL}/v1/models', timeout=5)
    ids = [m['id'] for m in r.json().get('data', [])]
    checks.append(('Modelo Llama disponível', len(ids) > 0, f'Modelos: {ids}'))
except Exception as e:
    checks.append(('Modelo Llama disponível', False, str(e)[:50]))

# Check 3: API funcional
try:
    r = client_vllm.chat.completions.create(
        model=MODEL_ID,
        messages=[{'role': 'user', 'content': 'Diga apenas: OK'}],
        temperature=0.0, max_tokens=10,
    )
    txt = r.choices[0].message.content.strip()
    checks.append(('API /v1/chat/completions funcional', len(txt) > 0, f'Resposta: "{txt[:30]}"'))
except Exception as e:
    checks.append(('API /v1/chat/completions funcional', False, str(e)[:60]))

# Check 4: Velocidade mínima
try:
    t0 = time.time()
    rb = client_vllm.chat.completions.create(
        model=MODEL_ID,
        messages=[{'role': 'user', 'content': 'Liste 3 artigos do Código Penal.'}],
        temperature=0.1, max_tokens=80,
    )
    dt  = time.time() - t0
    tks = rb.usage.completion_tokens / dt if dt > 0 else 0
    checks.append(('Velocidade de geração > 3 tok/s', tks > 3, f'{tks:.1f} tok/s'))
except Exception as e:
    checks.append(('Velocidade de geração > 3 tok/s', False, str(e)[:60]))

# Imprimir resultado
for nome, ok, detalhe in checks:
    icone = '✅' if ok else '❌'
    print(f'{icone} {nome}')
    print(f'   {detalhe}')

aprovados = sum(1 for _, ok, _ in checks if ok)
print(f'\n{"=" * 65}')
print(f'Resultado: {aprovados}/{len(checks)} checks aprovados')

if aprovados == len(checks):
    print('\n🎉 LAB 3 (versão Windows/Ollama) concluído com sucesso!')
    print('   Próximo: api_juridica_fastapi.py — API de produção com FastAPI!')
else:
    print('\n⚠️  Verifique se o Ollama está rodando e o modelo foi baixado.')
    print('   Terminal: ollama pull llama3.1:8b && ollama serve')